In [1]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

Load the JSON and create a master DataFrame of STIX objects

In [28]:
enterprise_file = "../enterprise-attack/enterprise-attack.json"
mobile_file = "../mobile-attack/mobile-attack.json"
ics_file = "../ics-attack/ics-attack.json"

with open(enterprise_file, "r", encoding="utf-8") as f:
    stix = json.load(f)

objects = stix.get("objects", [])
# Master dataframe (keeps the raw dict for reference)
master_df = pd.DataFrame([{
    "id": o.get("id"),
    "type": o.get("type"),
    "name": o.get("name"),
    "description": o.get("description"),
    "created": o.get("created"),
    "modified": o.get("modified"),
    "raw": o
} for o in objects])

master_df.head()


,id,type,name,description,created,modified,raw
0,x-mitre-collection--1f5f1533-f617-4ca8-9ab4-6a...,x-mitre-collection,Enterprise ATT&CK,ATT&CK for Enterprise provides a knowledge bas...,2018-01-17T12:56:55.080Z,2025-10-28T14:00:00.188Z,"{'type': 'x-mitre-collection', 'id': 'x-mitre-..."
1,x-mitre-matrix--eafc1b4c-5e56-4965-bd4e-66a6a8...,x-mitre-matrix,Enterprise ATT&CK,Below are the tactics and technique representi...,2018-10-17T00:14:20.652Z,2025-04-25T14:41:40.982Z,"{'type': 'x-mitre-matrix', 'spec_version': '2...."
2,course-of-action--00d7d21b-69d6-4797-88a2-c86f...,course-of-action,Password Filter DLL Mitigation,Ensure only valid password filters are registe...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:39.912Z,"{'type': 'course-of-action', 'spec_version': '..."
3,course-of-action--02f0f92a-0a51-4c94-9bda-6437...,course-of-action,Space after Filename Mitigation,Prevent files from having a trailing space aft...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:40.127Z,"{'type': 'course-of-action', 'spec_version': '..."
4,course-of-action--03c0c586-50ed-45a7-95f4-f496...,course-of-action,HISTCONTROL Mitigation,Prevent users from changing the <code>HISTCONT...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:40.291Z,"{'type': 'course-of-action', 'spec_version': '..."


Extract techniques (ATT&CK `attack-pattern` objects)

In [29]:
tech_objs = [o for o in objects if o.get("type") == "attack-pattern"]

def technique_row(o):
    # external id e.g. T1003 usually in external_references where source_name == 'mitre-attack'
    ext_refs = o.get("external_references", [])
    mitre_ref = next((r for r in ext_refs if r.get("source_name") == "mitre-attack"), {})
    external_id = mitre_ref.get("external_id")
    # kill_chain_phases may contain tactic phase names
    kcp = o.get("kill_chain_phases") or o.get("kill_chain_phases", []) or []
    phases = [p.get("phase_name") for p in kcp if isinstance(p, dict) and p.get("phase_name")]
    platforms = o.get("x_mitre_platforms") or o.get("x-mitre-platforms") or []
    data_sources = o.get("x_mitre_data_sources") or []
    return {
        "id": o.get("id"),
        "tech_name": o.get("name"),
        "external_id": external_id,
        "description": o.get("description"),
        "platforms": platforms,
        "kill_chain_phases": phases,
        "raw": o
    }

tech_df = pd.DataFrame([technique_row(o) for o in tech_objs])
tech_df.head()


,id,tech_name,external_id,description,platforms,kill_chain_phases,raw
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,Extra Window Memory Injection,T1055.011,Adversaries may inject malicious code into pro...,[Windows],"[defense-evasion, privilege-escalation]","{'type': 'attack-pattern', 'spec_version': '2...."
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,Scheduled Task,T1053.005,Adversaries may abuse the Windows Task Schedul...,[Windows],"[execution, persistence, privilege-escalation]","{'type': 'attack-pattern', 'spec_version': '2...."
2,attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...,Socket Filters,T1205.002,Adversaries may attach filters to a network so...,"[Linux, macOS, Windows]","[defense-evasion, persistence, command-and-con...","{'type': 'attack-pattern', 'spec_version': '2...."
3,attack-pattern--00d0b012-8a03-410e-95de-5826bf...,Indicator Removal from Tools,T1066,If a malicious tool is detected and quarantine...,"[Linux, macOS, Windows]",[defense-evasion],"{'type': 'attack-pattern', 'spec_version': '2...."
4,attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...,Archive via Utility,T1560.001,Adversaries may use utilities to compress and/...,"[Linux, macOS, Windows]",[collection],"{'type': 'attack-pattern', 'spec_version': '2...."


Extract tactics (from kill_chain_phases) — canonicalize into a DataFrame

In [30]:
# Many ATT&CK bundles don't provide tactic objects as separate 'x-mitre-tactic' entries,
# but techniques include kill_chain_phases referencing 'mitre-attack' phase_name (tactic).
# We'll get the unique list of tactics from the techniques kill_chain_phases:

tactics = sorted({phase for phases in tech_df["kill_chain_phases"].tolist() for phase in phases if phase})
tactics_df = pd.DataFrame({"tactic": tactics})
tactics_df


,tactic
0,collection
1,command-and-control
2,credential-access
3,defense-evasion
4,discovery
5,execution
6,exfiltration
7,impact
8,initial-access
9,lateral-movement


Relationships DataFrame (useful for group->technique and other links)

In [31]:
rel_objs = [o for o in objects if o.get("type") == "relationship"]

rel_df = pd.DataFrame([{
    "id": o.get("id"),
    "relationship_type": o.get("relationship_type"),
    "source_ref": o.get("source_ref"),
    "target_ref": o.get("target_ref"),
    "description": o.get("description"),
    "raw": o
} for o in rel_objs])

rel_df.head()


,id,relationship_type,source_ref,target_ref,description,raw
0,relationship--00038d0e-7fc7-41c3-9055-edb4d87e...,uses,malware--6a21e3a4-5ffe-4581-af9a-6a54c7536f44,attack-pattern--707399d6-ab3e-4963-9315-d9d381...,[Explosive](https://attack.mitre.org/software...,"{'type': 'relationship', 'spec_version': '2.1'..."
1,relationship--0005fb3b-274a-4ac1-8fb2-51366fcd...,mitigates,course-of-action--21da4fd4-27ad-4e9c-b93d-0b9b...,attack-pattern--43c9bc06-715b-42db-972f-52d25c...,Consider blocking download/transfer and execut...,"{'type': 'relationship', 'spec_version': '2.1'..."
2,relationship--000aa4d0-315e-40d7-b2b6-76e91ecf...,uses,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,[Indrik Spider](https://attack.mitre.org/group...,"{'type': 'relationship', 'spec_version': '2.1'..."
3,relationship--00192a5f-9dc0-445a-b010-d77bd08a...,uses,malware--425771c5-48b4-4ecd-9f95-74ed3fc9da59,attack-pattern--bf176076-b789-408e-8cba-7275e8...,[SombRAT](https://attack.mitre.org/software/S0...,"{'type': 'relationship', 'spec_version': '2.1'..."
4,relationship--001ecf24-8276-40d2-ba05-2d20e5c5...,uses,malware--b7010785-699f-412f-ba49-524da6033c76,attack-pattern--132d5b37-aac5-4378-a8dc-3127b1...,[GoldFinder](https://attack.mitre.org/software...,"{'type': 'relationship', 'spec_version': '2.1'..."


Map techniques to tactics (exploded rows, easy to group)

In [32]:
# explode kill_chain_phases into row-per-technique-per-tactic
tech_exploded = tech_df.explode("kill_chain_phases").rename(columns={"kill_chain_phases":"tactic"})
tech_exploded = tech_exploded[["id","external_id","tech_name","tactic","platforms"]]
tech_exploded.head()

,id,external_id,tech_name,tactic,platforms
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,T1055.011,Extra Window Memory Injection,defense-evasion,[Windows]
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,T1055.011,Extra Window Memory Injection,privilege-escalation,[Windows]
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,execution,[Windows]
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,persistence,[Windows]
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,privilege-escalation,[Windows]


## Analyses and Plots